In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def reshape_consumption_wide_to_long(raw_table: str):
    """
    Reshape raw consumption data from wide format to long format.

    The input bronze table is expected to contain one timestamp column
    (`measured_at`) and multiple group-specific consumption columns. This
    function converts the table into a long format where each row represents
    one `(group_id, timestamp_utc)` observation with its corresponding
    `target_consumption` value.

    Processing steps:
        - Reads the raw consumption table from the bronze layer
        - Casts all group-specific columns to double
        - Unpivots the wide table into long format using Spark SQL stack()
        - Casts group_id to int and target_consumption to double
        - Orders the result by group_id and timestamp_utc

    Args:
        raw_table: Name of the bronze-layer raw consumption table under
            fortum_challenge_data.01_bronze.

    Returns:
        A Spark DataFrame in long format with the columns:
            - timestamp_utc
            - group_id
            - target_consumption

    Side Effects:
        Reads a bronze-layer table using the global Spark session.
    """

    df = spark.table(f"fortum_challenge_data.01_bronze.{raw_table}")

    id_col = "measured_at"
    value_cols = [c for c in df.columns if c != id_col]

    # Cast all group_id columns to the same type first
    df_casted = df.select(
        F.col(id_col),
        *[F.col(c).cast("double").alias(c) for c in value_cols]
    )

    # Build stack expression
    stack_expr = ", ".join([f"'{c}', `{c}`" for c in value_cols])

    df_long = (
        df_casted
        .selectExpr(
            f"CAST({id_col} AS timestamp) AS timestamp_utc",
            f"stack({len(value_cols)}, {stack_expr}) AS (group_id, target_consumption)"
        )
        .withColumn("group_id", F.col("group_id").cast("int"))
        .withColumn("target_consumption", F.col("target_consumption").cast("double"))
        .orderBy("group_id", "timestamp_utc")
    )
    
    return df_long




def prepare_hourly_train_inference_actuals_baseline(df):
    """
    Prepare hourly consumption datasets for training, inference, actuals, and baseline comparison.

    The function creates lag-based and rolling-window features per group_id,
    removes early rows that do not have enough historical context for those
    features, and splits the dataset into four outputs:

        - training data before the final inference window
        - inference data for the final 48-hour prediction window
        - actual observed values for that 48-hour window
        - baseline comparison data from one week earlier

    Feature engineering:
        - consumption_lag_48
        - consumption_lag_168
        - consumption_lag_336
        - rolling_mean_168
        - rolling_std_168
        - lag_diff_48_168
        - lag_diff_168_336

    Current split logic:
        - Drops the first 336 rows per group_id
        - Inference window: 2024-09-29 00:00:00 to 2024-10-01 00:00:00
        - Baseline window: 2024-09-22 00:00:00 to 2024-09-24 00:00:00

    Args:
        df: Long-format hourly consumption Spark DataFrame, typically created by
            reshape_consumption_wide_to_long().

    Returns:
        A tuple of four Spark DataFrames:
            - df_train: hourly training data
            - df_inference: hourly inference data with target_consumption set to null
            - df_result: actual observed values for the inference window
            - df_baseline: baseline comparison data from the earlier window

    Side Effects:
        None beyond Spark DataFrame transformations.
    """
    window = Window.partitionBy("group_id").orderBy("timestamp_utc")

    rolling_window = window.rowsBetween(-168, -1)

    df_lags = df.withColumn("consumption_lag_48", F.lag("target_consumption", 48).over(window)) \
            .withColumn("consumption_lag_168", F.lag("target_consumption", 168).over(window)) \
            .withColumn("consumption_lag_336", F.lag("target_consumption", 336).over(window)) \
            .withColumn(
            "rolling_mean_168",
            F.avg("target_consumption").over(rolling_window)
        ) \
            .withColumn(
            "rolling_std_168",
            F.stddev("target_consumption").over(rolling_window)
        ) \
            .withColumn(
            "lag_diff_48_168",
            F.col("consumption_lag_48") - F.col("consumption_lag_168")
        ) \
            .withColumn(
            "lag_diff_168_336",
            F.col("consumption_lag_168") - F.col("consumption_lag_336")
        )

    # Add row index
    df_indexed = df_lags.withColumn(
        "row_num",
        F.row_number().over(window)
    )

    # Drop first 336 rows
    df_cut = df_indexed.filter(F.col("row_num") > 336).drop("row_num")

    df_asc = df_cut.orderBy("group_id", "timestamp_utc")

    cutoff_start = "2024-09-29 00:00:00"
    cutoff_end = "2024-10-01 00:00:00"

    baseline_start = "2024-09-22 00:00:00"
    baseline_end = "2024-09-24 00:00:00"


    # Historical/training part
    df_train = (
        df_asc
        .filter(F.col("timestamp_utc") < F.lit(cutoff_start).cast("timestamp"))
        .orderBy("group_id", "timestamp_utc")
    )

    # Final 48h inference part
    df_inference = (
        df_asc
        .filter(
            (F.col("timestamp_utc") >= F.lit(cutoff_start).cast("timestamp")) &
            (F.col("timestamp_utc") < F.lit(cutoff_end).cast("timestamp"))
        )
        .withColumn("target_consumption", F.lit(None).cast("double"))
        .orderBy("group_id", "timestamp_utc")
    )

    # Result table to compare the results to
    df_result = (
        df_asc
        .filter(
            (F.col("timestamp_utc") >= F.lit(cutoff_start).cast("timestamp")) &
            (F.col("timestamp_utc") < F.lit(cutoff_end).cast("timestamp"))
        )
        .orderBy("group_id", "timestamp_utc")
    )

    # Baseline table to compare the results to
    df_baseline = (
        df_asc
        .filter(
            (F.col("timestamp_utc") >= F.lit(baseline_start).cast("timestamp")) &
            (F.col("timestamp_utc") < F.lit(baseline_end).cast("timestamp"))
        )
        .orderBy("group_id", "timestamp_utc")
    )

    return df_train, df_inference, df_result, df_baseline




def prepare_monthly_train_inference(df):
    """
    Prepare monthly-horizon training and inference datasets.

    The function keeps the historical data as the monthly training set and
    generates a future hourly inference grid covering the next 12 months for
    every distinct group_id in the input data.

    Current monthly inference window:
        - Start: 2024-10-01 00:00:00
        - End: 2025-10-01 00:00:00 (exclusive)

    Args:
        df: Long-format consumption Spark DataFrame containing at least:
            - group_id
            - timestamp_utc
            - target_consumption

    Returns:
        A tuple of two Spark DataFrames:
            - df_train_12m: ordered historical training data
            - df_inference_12m: future hourly inference rows for each group_id,
              with target_consumption set to null

    Side Effects:
        Uses the global Spark session to generate a timestamp sequence.
    """
    cutoff_start = "2024-10-01 00:00:00"
    cutoff_end_12m = "2025-10-01 00:00:00"

    # Keep only unique group_ids
    df_groups = df.select("group_id").distinct()

    # Generate hourly future timestamps
    df_timestamps = (
        spark.sql(f"""
            SELECT explode(
                sequence(
                    timestamp('{cutoff_start}'),
                    timestamp('{cutoff_end_12m}') - interval 1 hour,
                    interval 1 hour
                )
            ) AS timestamp_utc
        """)
    )

    # Cross join: every group_id gets every future timestamp
    df_inference_12m = (
        df_groups
        .crossJoin(df_timestamps)
        .withColumn("target_consumption", F.lit(None).cast("double"))
        .orderBy("group_id", "timestamp_utc")
    )

    df_train_12m = df.orderBy("group_id", "timestamp_utc")

    return df_train_12m, df_inference_12m




def write_dataframe_to_layer_table(df, schema_name, table_name: str):
    """
    Write a Spark DataFrame to a specified medallion-layer table.

    The function overwrites the destination table under the given schema
    in the fortum_challenge_data catalog.

    Args:
        df: Spark DataFrame to write.
        schema_name: Target schema name, for example "02_silver" or "04_results".
        table_name: Target table name within the schema.

    Returns:
        None.

    Side Effects:
        - Overwrites any existing data in the destination table.
        - Writes the DataFrame to the metastore.
        - Prints a success message to standard output.
    """
    df.write \
        .mode("overwrite") \
        .saveAsTable(f"fortum_challenge_data.{schema_name}.{table_name}")
    print(f"Successfully wrote to {schema_name} table {table_name}")



if __name__ == "__main__":

    raw_table = "consumption_data_raw"

    wide_table = reshape_consumption_wide_to_long(raw_table)

    hourly_training_inference_table = prepare_hourly_train_inference_actuals_baseline(wide_table)

    monthly_training_inference_table = prepare_monthly_train_inference(wide_table)

    silver = "02_silver"
    results = "04_results"

    clean_hourly_training_table = "consumption_hourly_training_clean"
    clean_hourly_inference_table = "consumption_hourly_inference_clean"
    clean_monthly_training_table = "consumption_monthly_training_clean"
    clean_monthly_inference_table = "consumption_monthly_inference_clean"
    clean_hourly_actuals_table = "consumption_hourly_actuals"
    clean_hourly_baseline_table = "consumption_hourly_baseline"

    write_dataframe_to_layer_table(hourly_training_inference_table[0], silver, clean_hourly_training_table)
    write_dataframe_to_layer_table(hourly_training_inference_table[1], silver, clean_hourly_inference_table)
    write_dataframe_to_layer_table(monthly_training_inference_table[0], silver, clean_monthly_training_table)
    write_dataframe_to_layer_table(monthly_training_inference_table[1], silver, clean_monthly_inference_table)
    write_dataframe_to_layer_table(hourly_training_inference_table[2], results, clean_hourly_actuals_table)
    write_dataframe_to_layer_table(hourly_training_inference_table[3], results, clean_hourly_baseline_table)

    hourly_training_inference_table[0].display()